<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/04_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.6.0?labpath=examples%2F04_export.ipynb)

# 04 — Framework Export

Export circuits to **OpenQASM 3.0, Qiskit, PennyLane, Cirq, and TKET**.
Frameworks without the optional dependency installed are skipped automatically.

```bash
pip install quprep[qiskit]
pip install quprep[pennylane]
pip install quprep[cirq]
pip install quprep[tket]
```

In [ ]:
import numpy as np

import quprep as qd

rng = np.random.default_rng(42)
X = rng.uniform(0, 1, size=(5, 3))

# Single encoded result for per-framework demos
enc = qd.AngleEncoder(rotation="ry").encode(np.array([0.3, 1.1, 0.7]) * np.pi)

## 1. OpenQASM 3.0 — no dependencies

Universal interchange format accepted by all major frameworks and hardware backends.

In [ ]:
exp = qd.QASMExporter()
print(exp.export(enc))

## 2. Batch save — write each sample to its own file

In [ ]:
result = qd.prepare(X, encoding="angle")
paths = exp.save_batch(result.encoded, "/tmp/quprep_batch/", stem="circuit")

print(f"Saved {len(paths)} files to {paths[0].parent}/")
for p in paths:
    print(f"  {p.name}")

## 3. Qiskit

Returns a `qiskit.QuantumCircuit`. Requires `pip install quprep[qiskit]`.

In [ ]:
try:
    from quprep.export.qiskit_export import QiskitExporter

    qc = QiskitExporter().export(enc)
    print(qc.draw(output="text"))
    print(f"num_qubits : {qc.num_qubits}")
    print(f"depth      : {qc.depth()}")
except ImportError as e:
    print(f"skipped — {e}")

## 4. Amplitude encoding via Qiskit (StatePreparation)

In [ ]:
try:
    from quprep.export.qiskit_export import QiskitExporter

    unit_vec = np.array([0.5, 0.5, 0.5, 0.5])
    enc_amp = qd.AmplitudeEncoder().encode(unit_vec)
    qc_amp = QiskitExporter().export(enc_amp)
    print(qc_amp.draw(output="text"))
except ImportError as e:
    print(f"skipped — {e}")

## 5. PennyLane

Returns a callable `qml.QNode`. Requires `pip install quprep[pennylane]`.

In [ ]:
try:
    from quprep.export.pennylane_export import PennyLaneExporter

    qnode = PennyLaneExporter().export(enc)
    state = qnode()
    print(f"QNode  : {qnode}")
    print(f"State  : {state}")
except ImportError as e:
    print(f"skipped — {e}")

## 6. Cirq

Returns a `cirq.Circuit`. Requires `pip install quprep[cirq]`.

In [ ]:
try:
    from quprep.export.cirq_export import CirqExporter

    circuit = CirqExporter().export(enc)
    print(circuit)
    print(f"qubits : {sorted(circuit.all_qubits())}")
except ImportError as e:
    print(f"skipped — {e}")

## 7. TKET / pytket

Returns a `pytket.Circuit`. Requires `pip install quprep[tket]`.

In [ ]:
try:
    from quprep.export.tket_export import TKETExporter

    circuit = TKETExporter().export(enc)
    print(f"n_qubits : {circuit.n_qubits}")
    print(f"n_gates  : {circuit.n_gates}")
    print(f"depth    : {circuit.depth()}")
except ImportError as e:
    print(f"skipped — {e}")